# Train Model

In [70]:
import torch
import torch.nn as nn
import torch.optim as optim
from copy import deepcopy

torch.set_num_threads(1)

In [71]:
import pandas as pd

In [72]:
DATA_FILE_PATH = "./datasets/combined_dataset.csv"
data = pd.read_csv(DATA_FILE_PATH)

In [73]:
data.head()

,is_scara,is_ned,q1,q2,q3,x,y,z
0,1,0,0.341700,-0.788656,-0.052945,0.577740,-0.005921,0.129027
1,0,1,1.078752,0.192634,0.761283,-0.204516,0.109663,0.206201
2,0,1,-2.652503,0.264075,-0.374363,0.146819,-0.276348,0.450027
3,0,1,1.190232,-0.436801,-0.931734,0.067656,-0.026666,0.629680
4,1,0,0.147300,0.481437,-0.117415,0.574681,0.196077,0.064557


In [74]:
X = data[["is_scara", "is_ned", "x", "y", "z"]]
y = data[["q1", "q2", "q3"]]

In [75]:
X.head()

,is_scara,is_ned,x,y,z
0,1,0,0.577740,-0.005921,0.129027
1,0,1,-0.204516,0.109663,0.206201
2,0,1,0.146819,-0.276348,0.450027
3,0,1,0.067656,-0.026666,0.629680
4,1,0,0.574681,0.196077,0.064557


In [76]:
y.head()

,q1,q2,q3
0,0.341700,-0.788656,-0.052945
1,1.078752,0.192634,0.761283
2,-2.652503,0.264075,-0.374363
3,1.190232,-0.436801,-0.931734
4,0.147300,0.481437,-0.117415


In [77]:
from sklearn.model_selection import train_test_split

In [78]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.125, random_state=42, stratify=X_train["is_scara"])

In [79]:
X_train.shape, X_val.shape, X_test.shape

((35000, 5), (5000, 5), (10000, 5))

In [80]:
y_train.shape, y_val.shape, y_test.shape

((35000, 3), (5000, 3), (10000, 3))

# Scale data

In [81]:
from sklearn.preprocessing import StandardScaler

input_scaler = StandardScaler()
target_scaler = StandardScaler()

X_train_scaled = input_scaler.fit_transform(X_train)
y_train_scaled = target_scaler.fit_transform(y_train)

X_val_scaled = input_scaler.transform(X_val)
y_val_scaled = target_scaler.transform(y_val)
X_test_scaled = input_scaler.transform(X_test)

X_train_ts = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_ts = torch.tensor(y_train_scaled, dtype=torch.float32)
X_val_ts = torch.tensor(X_val_scaled, dtype=torch.float32)
y_val_ts = torch.tensor(y_val_scaled, dtype=torch.float32)
X_test_ts = torch.tensor(X_test_scaled, dtype=torch.float32)

In [82]:
class InverseKinematicNet(nn.Module):
    def __init__(self, input_size, output_size):
        super(InverseKinematicNet, self).__init__()

        self.layer1 = nn.Linear(input_size, 128)
        self.layer2 = nn.Linear(128, 256)
        self.layer3 = nn.Linear(256, 128)
        self.layer4 = nn.Linear(128, output_size)

        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)

        x = self.layer2(x)
        x = self.relu(x)

        x = self.layer3(x)
        x = self.relu(x)

        out = self.layer4(x)

        return out

In [83]:
torch.manual_seed(42)
model = InverseKinematicNet(input_size=5, output_size=3)

In [84]:
loss_function = nn.MSELoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=15)

In [85]:
epochs = 500
batch_size = 512
best_loss = float("inf")
wait = 0

for epoch in range(epochs):
    model.train()
    indices = torch.randperm(len(X_train_ts))
    train_loss = 0

    for batch in indices.split(batch_size):
        outputs = model(X_train_ts[batch])
        loss = loss_function(outputs, y_train_ts[batch])

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(batch)

    model.eval()
    with torch.no_grad():
        val_loss = loss_function(model(X_val_ts), y_val_ts).item()

    scheduler.step(val_loss)

    if val_loss < best_loss:
        best_loss = val_loss
        best_model = deepcopy(model.state_dict())
        wait = 0
    else:
        wait += 1

    if (epoch + 1) % 20 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {train_loss / len(X_train_ts):.4f}, Val Loss: {val_loss:.4f}")

    if wait >= 60:
        break

model.load_state_dict(best_model)

Epoch [20/500], Loss: 0.2730, Val Loss: 0.2703
Epoch [40/500], Loss: 0.2624, Val Loss: 0.2641
Epoch [60/500], Loss: 0.2539, Val Loss: 0.2569
Epoch [80/500], Loss: 0.2470, Val Loss: 0.2496
Epoch [100/500], Loss: 0.2445, Val Loss: 0.2512
Epoch [120/500], Loss: 0.2407, Val Loss: 0.2458
Epoch [140/500], Loss: 0.2403, Val Loss: 0.2455
Epoch [160/500], Loss: 0.2382, Val Loss: 0.2441
Epoch [180/500], Loss: 0.2366, Val Loss: 0.2436
Epoch [200/500], Loss: 0.2360, Val Loss: 0.2436
Epoch [220/500], Loss: 0.2358, Val Loss: 0.2434
Epoch [240/500], Loss: 0.2355, Val Loss: 0.2431
Epoch [260/500], Loss: 0.2352, Val Loss: 0.2432
Epoch [280/500], Loss: 0.2352, Val Loss: 0.2432
Epoch [300/500], Loss: 0.2352, Val Loss: 0.2432


<All keys matched successfully>

In [86]:
model.eval()
with torch.no_grad():
    y_pred = model(X_test_ts)

In [87]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

y_test_np = y_test.to_numpy()
y_pred_np = target_scaler.inverse_transform(y_pred.cpu().numpy())

mse = mean_squared_error(y_test_np, y_pred_np)
mae = mean_absolute_error(y_test_np, y_pred_np)
r2 = r2_score(y_test_np, y_pred_np)

print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"R2 Score: {r2:.4f}")

Mean Squared Error (MSE): 0.1532
Mean Absolute Error (MAE): 0.1623
R2 Score: 0.7491


## Exporting Models

In [88]:
import joblib
from pathlib import Path

Path("./Models").mkdir(exist_ok=True)

In [89]:
joblib.dump(input_scaler, "./Models/input_scaler.joblib")
joblib.dump(target_scaler, "./Models/target_scaler.joblib")

['./Models/target_scaler.joblib']

In [90]:
from safetensors.torch import save_file

save_file(model.state_dict(), "./Models/model.safetensors")

In [91]:
my_X_data = X_test.iloc[[0]]
my_X_data_scaled = input_scaler.transform(my_X_data)
my_X_data_tf = torch.tensor(my_X_data_scaled, dtype=torch.float32)

In [92]:
model.eval()
with torch.no_grad():
    my_y_pred = model(my_X_data_tf)

In [93]:
my_y_pred = target_scaler.inverse_transform(my_y_pred.cpu().numpy())

In [94]:
pd.DataFrame(my_y_pred, columns=y.columns, index=my_X_data.index)

,q1,q2,q3
33553,-0.617122,0.075704,-0.134064


In [95]:
y_test.loc[my_X_data.index]

,q1,q2,q3
33553,-0.549183,-0.023518,-0.131555


In [96]:
my_X_data = [[0, 1, 0.0000, 0.1815, 0.2045]]
my_X_data_scaled = input_scaler.transform(my_X_data)
my_X_data_tf = torch.tensor(my_X_data_scaled, dtype=torch.float32)

/Users/sakibdalal/Projects/Robotics-01/Notebook/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2830: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [97]:
my_X_data_tf

tensor([[-1.0014,  1.0014, -0.9247,  0.8988, -0.4452]])

In [98]:
model.eval()
with torch.no_grad():
    my_y_pred = model(my_X_data_tf)

In [99]:
my_y_pred = target_scaler.inverse_transform(my_y_pred.cpu().numpy())

In [100]:
my_y_pred

array([[-0.01851335,  0.0517611 ,  0.952636  ]], dtype=float32)